# WHW09 starter notebook: coupled oscillators with NumPy's eigenvalue solver

Two equal masses $m$ sit in a line, joined by three identical springs $k$:
wall -- spring -- mass 1 -- spring -- mass 2 -- spring -- wall. Call the
displacements from equilibrium $x_1$ and $x_2$. Newton's second law for each
mass gives

$$x_1'' = -\frac{2k}{m} x_1 + \frac{k}{m} x_2, \qquad x_2'' = \frac{k}{m} x_1 - \frac{2k}{m} x_2 ,$$

or, as a matrix equation in the form we used in class,

$$\mathbf{x}'' = M \mathbf{x}, \qquad M = \frac{k}{m}\begin{pmatrix} -2 & 1 \\ 1 & -2 \end{pmatrix}.$$

Same convention as class: a normal mode is an eigenvector of $M$, and its
eigenvalue is $\lambda = -\omega^2$, so the mode swings at $\omega = \sqrt{-\lambda}$.
Throughout, set $k/m = 1$ (so $t$ is in seconds when $k/m$ is in s$^{-2}$).

Here's the problem:

1. **Before computing:** describe the two normal modes physically (how do the
   masses move relative to each other?), say which one you expect to have the
   higher frequency, and give a one-sentence physical reason.
2. Use `np.linalg.eig` on $M$ and compare the eigenvalues and eigenvectors with
   your prediction. Note anything that surprised you about how NumPy normalized
   or ordered the eigenvectors.
3. Plot $x_1(t)$ and $x_2(t)$ on $0 \le t \le 20$ for two starting conditions, both
   released from rest: **(a)** displacement equal to the first eigenvector NumPy
   gives you, and **(b)** displacement $(1, 0)$.
4. **(c)** Do (b) again with the middle spring ten times weaker than the outer
   two (the matrix is in section 7), on $0 \le t \le 150$.
5. Write two or three sentences on why plot (b) shows *beating* -- energy
   sloshing back and forth between the two masses -- while plot (a) does not,
   and why the beating is so much cleaner in (c).

We have *not* used NumPy's eigenvalue solver before, and we haven't talked
about beating, so this notebook shows you each new piece on a *different*
example first, then leaves this problem to you.

**How this is graded.** Unlike the practice lists on the WHW page, this problem is turned in. It should take 20-40 minutes. There are three main parts:

1. **Predict:** write down what you expect *before* you run anything, and leave the prediction (especially if it's wrong!) in your writeup.
2. **Interpret:** two or three sentences on what the output actually shows.
3. **Report a dead end:** one sentence on something you tried that did not work.

This is graded the same good-faith way as the rest of the WHW, and those three moves are what good faith looks like here: a messy writeup that has all three is a complete submission, and a polished one missing them is not. The record of your own thinking is the assignment.

## 0. Setup

Same setup as always: matplotlib for plotting, sympy for symbolic manipulation,
and numpy for numerical python calculations.

In [ ]:
%matplotlib inline
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import *
init_printing()

## 1. Predict first (before you run anything)

1. Describe the two normal modes physically. In each one, how do the two masses
   move relative to each other?
2. Which mode has the higher frequency? Give a one-sentence physical reason
   (think about which springs get stretched in each mode).

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 2. NumPy's eigenvalue solver, and how to read what it gives you

Let's use the three springs from class 23 (Mon Nov 2), where we already know the
answer:

$$M_\text{demo} = \begin{pmatrix} -4 & 3 \\ 2/3 & -3 \end{pmatrix}, \qquad
\lambda = -2 \text{ with } (1, 2/3), \qquad \lambda = -5 \text{ with } (1, -1/3).$$

We make the matrix with `np.array`, as in the WHW08 starter (each inner list is
one *row*). Then `np.linalg.eig` does the whole eigenvalue problem for us.

In [ ]:
M_demo = np.array([[-4, 3],
                   [2/3, -3]])
eigenvalues_demo, eigenvectors_demo = np.linalg.eig(M_demo)
eigenvalues_demo = eigenvalues_demo.real
eigenvectors_demo = eigenvectors_demo.real
print('eigenvalues:')
print(eigenvalues_demo)
print('eigenvectors (as columns):')
print(eigenvectors_demo)

Here's how to read that, line by line:

- `np.linalg.eig(M_demo)` hands back **two** things, so we catch them in two
  names, `eigenvalues_demo, eigenvectors_demo = ...`.
- Depending on the NumPy version, `eig` may hand everything back as complex
  numbers, printed with a `+0.j` tacked on. Our eigenvalues are real, so
  `.real` keeps just the real part and makes the printout readable.
- `eigenvalues_demo` is a plain array of the eigenvalues. NumPy does **not**
  sort them: they come out in whatever order its algorithm finds them.
- `eigenvectors_demo` is a 2x2 array whose **columns** are the eigenvectors
  (columns, not rows! Same idea as the WHW08 starter, where points were
  columns.) Column `k` goes with eigenvalue `k`.
- To grab column `k`, write `eigenvectors_demo[:, k]`. The `:` means "every
  row", so `[:, 0]` is "every row of column 0", the first eigenvector.
- NumPy scales every eigenvector to **length 1**, and it may flip its sign. Any
  multiple of an eigenvector is an eigenvector (class 22), so that's fine, but it
  means you won't see $(1, 2/3)$ printed. Dividing by the first entry gets us back
  to the form we used in class.

Let's check all of that: $M\mathbf{v}$ should equal $\lambda \mathbf{v}$ for each pair
(the `@` is matrix multiplication, as in the WHW08 starter).

In [ ]:
for k in range(2):
    lam = eigenvalues_demo[k]
    v = eigenvectors_demo[:, k]
    print('eigenvalue', lam)
    print('   M @ v      =', M_demo @ v)
    print('   lambda * v =', lam * v)
    print('   length of v =', np.sqrt(v[0]**2 + v[1]**2))
    print('   v rescaled so the first entry is 1:', v / v[0])
    print('   omega = sqrt(-lambda) =', np.sqrt(-lam))

## 3. From normal modes to motion

From class 24 (Wed Nov 4): the general motion is a sum of the two normal modes.
If the masses are **released from rest**, each mode is a cosine:

$$\mathbf{x}(t) = c_1 \mathbf{v}_1 \cos(\omega_1 t) + c_2 \mathbf{v}_2 \cos(\omega_2 t).$$

At $t = 0$ the cosines are 1, so $\mathbf{x}(0) = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2$.
Writing the eigenvectors as the columns of a matrix $V$ (which is exactly what
`np.linalg.eig` gave us!) that says $\mathbf{x}(0) = V \mathbf{c}$, so
$\mathbf{c} = V^{-1}\mathbf{x}(0)$: the inverse of the mode matrix, as in class 21's 6.115.
NumPy's inverse is `np.linalg.inv`.

Here it is for the demo springs, released from rest at $x_1 = 5$, $x_2 = 7$:

- `np.linalg.inv(eigenvectors_demo) @ x0` is $V^{-1}\mathbf{x}(0)$: how much of each mode we need.
- `omegas = np.sqrt(-eigenvalues)` turns each eigenvalue into its frequency.
- The `for` loop (as in the WHW04 starter) adds up the two modes. Row 0 of the
  eigenvector matrix is mass 1's share of each mode, row 1 is mass 2's share.
- `x1_vals` and `x2_vals` start as arrays of zeros, one slot per time, and
  each pass through the loop adds one mode's contribution at *every* time at
  once (element-wise, as in the WHW06 starter).

In [ ]:
def motion_demo(x0, t_vals):
    eigenvalues, eigenvectors = np.linalg.eig(M_demo)
    eigenvalues = eigenvalues.real
    eigenvectors = eigenvectors.real
    omegas = np.sqrt(-eigenvalues)
    amounts = np.linalg.inv(eigenvectors) @ x0
    x1_vals = np.zeros(len(t_vals))
    x2_vals = np.zeros(len(t_vals))
    for k in range(2):
        x1_vals = x1_vals + amounts[k] * eigenvectors[0, k] * np.cos(omegas[k] * t_vals)
        x2_vals = x2_vals + amounts[k] * eigenvectors[1, k] * np.cos(omegas[k] * t_vals)
    return x1_vals, x2_vals

Here's a plotting function we'll reuse for every plot in this notebook, and the
demo's motion. Mass 2 is drawn dashed, so you can still see it if it lands
exactly on top of mass 1. Check: at $t = 0$ the curves should start at 5 and 7.

In [ ]:
def plot_two_masses(t_vals, x1_vals, x2_vals, title):
    plt.figure(figsize=(9, 4))
    plt.plot(t_vals, x1_vals, label='$x_1$ (mass 1)')
    plt.plot(t_vals, x2_vals, '--', label='$x_2$ (mass 2)')
    plt.xlabel('t (s)')
    plt.ylabel('displacement')
    plt.title(title)
    plt.legend()
    plt.show()

t_demo = np.linspace(0, 20, 2000)
x1_demo, x2_demo = motion_demo(np.array([5, 7]), t_demo)
print('start:', x1_demo[0], x2_demo[0])
plot_two_masses(t_demo, x1_demo, x2_demo, 'class 23 springs, released from rest at (5, 7)')

## 4. A new idea: beating

What happens when you add two cosines with *nearby* frequencies? There's a
trig identity for it:

$$\cos a + \cos b = 2 \cos\!\left(\frac{a - b}{2}\right) \cos\!\left(\frac{a + b}{2}\right), \qquad
\cos a - \cos b = -2 \sin\!\left(\frac{a + b}{2}\right) \sin\!\left(\frac{a - b}{2}\right).$$

(Check the first one: expand $\cos(A + B) + \cos(A - B)$ with $A = (a+b)/2$ and
$B = (a-b)/2$.)

Put $a = \omega_1 t$ and $b = \omega_2 t$. The factor with $(\omega_1 + \omega_2)/2$ wiggles
at about the average frequency. The factor with $(\omega_1 - \omega_2)/2$ changes
**slowly** when the two frequencies are close. So the sum looks like a fast
oscillation whose size swells and shrinks slowly: the slow factor is an
**envelope**. That's beating. (It's what you hear when two guitar strings are
almost, but not quite, in tune: a "wah-wah-wah" at the difference frequency.)

Here it is for $\cos(1.0\,t) + \cos(1.1\,t)$, with the envelope $\pm 2\cos(0.05\,t)$ drawn on top.

In [ ]:
t_beat = np.linspace(0, 200, 4000)
y_beat = np.cos(1.0 * t_beat) + np.cos(1.1 * t_beat)
envelope = 2 * np.cos(0.05 * t_beat)

plt.figure(figsize=(9, 4))
plt.plot(t_beat, y_beat, label='cos(1.0 t) + cos(1.1 t)')
plt.plot(t_beat, envelope, 'k--', label='envelope: $\\pm 2\\cos(0.05\\,t)$')
plt.plot(t_beat, -envelope, 'k--')
plt.xlabel('t (s)')
plt.legend()
plt.show()

The fast wiggle has period $2\pi/1.05 \approx 6$ s; the envelope goes through
zero every $\pi/0.05 \approx 63$ s. The closer the two frequencies, the slower the
envelope compared with the wiggle, and the cleaner the beats look.

## 5. Predict again (before you plot)

You're about to release the masses from rest at $(1, 0)$: mass 1 pulled
aside, mass 2 at equilibrium. Sketch (in words) what you expect $x_1(t)$ and
$x_2(t)$ to do. Does mass 2 ever move as much as mass 1 did at the start? What
changes if the middle spring is much weaker?

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 6. Your turn: the eigenvalues and eigenvectors

In the cell below, change the matrix to our $M$ (with $k/m = 1$) from the top of
the notebook. Then run it and the check cell after it.

In [ ]:
# Change the line below.
my_M = np.array([[-4, 3], [2/3, -3]])

my_eigenvalues, my_eigenvectors = np.linalg.eig(my_M)
my_eigenvalues = my_eigenvalues.real
my_eigenvectors = my_eigenvectors.real
print('eigenvalues:')
print(my_eigenvalues)
print('eigenvectors (as columns):')
print(my_eigenvectors)

In [ ]:
for k in range(2):
    v = my_eigenvectors[:, k]
    print('eigenvalue', my_eigenvalues[k], ' omega =', np.sqrt(-my_eigenvalues[k]))
    print('   eigenvector', v, '  rescaled so the first entry is 1:', v / v[0])

## 7. Your turn: the motion

Copy of the demo's `motion_demo`, renamed `my_motion`. One line changes: it should use your matrix.

Then the next cell makes all three plots for you:

- **(a)** from rest at the first eigenvector NumPy gave you, `my_eigenvectors[:, 0]`, on $0 \le t \le 20$;
- **(b)** from rest at $(1, 0)$, on $0 \le t \le 20$;
- **(c)** from rest at $(1, 0)$, but with the middle spring ten times weaker than the
  outer two ($k_\text{middle} = k/10$), on $0 \le t \le 150$. Redo Newton's law if you like;
  the matrix is
  $$M_\text{weak} = \frac{k}{m}\begin{pmatrix} -1.1 & 0.1 \\ 0.1 & -1.1 \end{pmatrix}.$$
  For (c), `my_motion` takes the matrix as an input, so the plotting cell just hands it
  `M_weak` instead of `my_M`.

In [ ]:
def my_motion(M, x0, t_vals):
    # Change the line below.
    eigenvalues, eigenvectors = np.linalg.eig(M_demo)
    eigenvalues = eigenvalues.real
    eigenvectors = eigenvectors.real
    omegas = np.sqrt(-eigenvalues)
    amounts = np.linalg.inv(eigenvectors) @ x0
    x1_vals = np.zeros(len(t_vals))
    x2_vals = np.zeros(len(t_vals))
    for k in range(2):
        x1_vals = x1_vals + amounts[k] * eigenvectors[0, k] * np.cos(omegas[k] * t_vals)
        x2_vals = x2_vals + amounts[k] * eigenvectors[1, k] * np.cos(omegas[k] * t_vals)
    return x1_vals, x2_vals

In [ ]:
t_short = np.linspace(0, 20, 2000)
t_long = np.linspace(0, 150, 6000)
M_weak = np.array([[-1.1, 0.1], [0.1, -1.1]])

x0_a = my_eigenvectors[:, 0]
x1_a, x2_a = my_motion(my_M, x0_a, t_short)
plot_two_masses(t_short, x1_a, x2_a, f'(a) from rest at the first eigenvector, {np.round(x0_a, 3)}')

x0_b = np.array([1, 0])
x1_b, x2_b = my_motion(my_M, x0_b, t_short)
plot_two_masses(t_short, x1_b, x2_b, '(b) from rest at (1, 0)')

x1_c, x2_c = my_motion(M_weak, x0_b, t_long)
plot_two_masses(t_long, x1_c, x2_c, '(c) weak middle spring, from rest at (1, 0)')

## 8. What you found

**(a)** Compare the eigenvalues and eigenvectors NumPy gave you with your
prediction from section 1. Which mode is higher frequency? Anything that
surprised you about how NumPy normalized or ordered the eigenvectors?

*(your answer here)*

**(b)** In two or three sentences: why does plot (b) show beating -- energy
sloshing back and forth between the two masses -- while plot (a) does not? Why
is the beating so much cleaner in plot (c)? (Section 4's identities will help.)

*(your answer here)*

**(c)** One thing that did not work along the way (a row where you wanted a
column, a sign flip, a plot that started in the wrong place, ...).

*(your answer here)*